`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`

In [ ]:
library(BioCircos)
library(dplyr)
library(DT)
library(htmltools)

# General Stats

In [ ]:
fai <- params$faidx
statsfile <- params$statsfile
samplename <- params$sample

In [ ]:
sv <- read.table(statsfile, header = T,
  colClasses = c("factor", "factor", "integer", "integer", "character", "factor", "integer", "integer")
)
if (nrow(sv) == 0) {
  cat(paste0("There are no variants in the file ", "`", samplename, ".bcf`"))
  knitr::knit_exit()
}

In [ ]:
summinfo <- sv %>%
  group_by(contig, type) %>%
  summarise(count = n()) %>% 
  group_by(type) %>%
  summarise(
    total = sum(count),
    avg_per_contig = round(mean(count), 2),
    sd = round(sd(count), 2)
)

nvar <- sum(summinfo$total)

##

In [ ]:
list(
  color = ifelse(nvar > 0, "success", "warning"),
  value = prettyNum(nvar, big.mark = ",")
)

In [ ]:
nbnd <- ifelse("BND" %in% summinfo$type, summinfo$total[which(summinfo$type == "BND")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(nbnd, big.mark = ",")
)

In [ ]:
ndel <- ifelse("DEL" %in% summinfo$type, summinfo$total[which(summinfo$type == "DEL")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(ndel, big.mark = ",")
)

In [ ]:
ndup <- ifelse("DUP" %in% summinfo$type, summinfo$total[which(summinfo$type == "DUP")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(ndup, big.mark = ",")
)

In [ ]:
ninv <- ifelse("INV" %in% summinfo$type, summinfo$total[which(summinfo$type == "INV")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(ninv, big.mark = ",")
)

##
This report details the structural variants identified by [LEVIATHAN](https://github.com/morispi/LEVIATHAN)
[(preprint)](https://doi.org/10.1101/2021.03.25.437002). This tab (`General Stats`) shows overview
information. The `Per-Contig Plot` tab shows you an interactive
plot detailing all variants identified. The variants are given by:

INV (Inversion)
: A segment that broke off and reattached within the same chromosome, but in reverse orientation

DUP (Duplication)
: A type of mutation that involves the production of one or more copies of a gene or region of a chromosome

DEL (Deletion)
: A type of mutation that involves the loss of one or more nucleotides from a segment of DNA

BND (Breakend)
: The endpoint of a structural variant, which can be useful to explain complex variants

##

In [ ]:
sv$length <- gsub("\\.", "1", sv$length) %>%  as.numeric()

## Various Stats Tables
::: {.card title="All variants found"}
This table details all the variants LEVIATHAN detected and evidence for those detections.

In [ ]:
DT::datatable(
  sv,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = paste0(samplename , "_sv_leviathan`"))),  
    scrollX = T,
    paging = T
  )
)

:::
::: {.card title="Variants per Contig"}
This table details each type of structural variant for every contig. 

In [ ]:
grpstats <- sv %>%
  group_by(contig, type) %>%
  summarise(count = n(), total_bp  = sum(length))

DT::datatable(
  grpstats, 
  rownames = F, 
  filter = "top", 
  extensions = 'Buttons', 
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = paste0(samplename ,"_sv_leviathan_per_contig"))),
    scrollX = TRUE,
    paging = T
  )
)

:::

# Per-Contig Plot

In [ ]:
fa.sizes <- read.table(fai, header = F) %>% arrange(desc(2))
colnames(fa.sizes) <- c("contig", "size")

plot_contigs <- params$contigs
if (all(plot_contigs == "default")){
  # make sure that the contigs with SV are the ones being plotted, not the others
  fa.sizes <- fa.sizes[fa.sizes$contig %in% unique(sv$contig), 1:2]
  # limit the data to only the 30 of the largest present contigs
  fa.sizes <- fa.sizes[1:min(nrow(fa.sizes), 30), ]
} else {
  fa.sizes <- fa.sizes[fa.sizes$contig %in% plot_contigs, 1:2]
}

# filter out variants that aren't on one of these contigs
sv <- sv[sv$contig %in% fa.sizes$contig, ]
sv$contig <- factor(sv$contig)

##
To the right is a circular plot to visualize the distribution of _putative_ structural variants
across (up to) 30 of the largest contigs, or whichever contigs were provided. If you are unfamiliar with this kind of visualization,
it's a circular representation of a linear genome. Each "wedge" is a different contig, from position 0 to 
the end of the contig, and is labelled by the contig name. Each internal (grey) ring is a plot
of observed structural variants for that contig. The legend shows which colors correspond to which type of variant. To save the plot,
you will need to take a screenshot. Very small variants may be difficult to see or
may not appear on the plot. Deletions are shown as points as they tend to be small
and clump together.

##
::: {.card width=15% title="Legend"}

In [ ]:
calculate_luminance <- function(hex) {
  rgb_values <- col2rgb(hex) / 255
  luminance <- 0.2126 * rgb_values[1] + 0.7152 * rgb_values[2] + 0.0722 * rgb_values[3]
  return(luminance)
}

# Function to decide text color based on background luminance
get_text_color <- function(background_color) {
  luminance <- calculate_luminance(background_color)
  if (luminance > 0.5) {
    # Dark background -> Light text (black)
    return("black") 
  } else {
    # Light background -> Dark text (white)
    return("white") 
  }
}
color.palette <- c("deletion" = "#5a8c84", "duplication" = "#99278a", "inversion" = "#4a9fea", "breakend" = "#c56e34")
DT::datatable(
  data.frame(Variant = c("breakend","deletion","duplication", "inversion")),
  rownames = F,
  fillContainer = T,
  colnames = "Variant Type",
  options = list(
    dom = 't',
    scrollX = F,
    scrollY = F,
    paging = F
  )
) %>% formatStyle(
  "Variant",
  backgroundColor = styleEqual(names(color.palette), color.palette),
  # Adjust text color dynamically
  color = styleEqual(names(color.palette), sapply(color.palette, get_text_color))
)

:::

###

In [ ]:
genomeChr <- fa.sizes$size
names(genomeChr) <- fa.sizes$contig
genomeChr <- as.list(genomeChr)
tracks <- BioCircosTracklist()
radius <- 0.3

# Add one track for each SV type
# inversions
inv <- sv[sv$type == "INV",]
if(nrow(inv) > 0){
  tracks <- tracks + BioCircosArcTrack(
    'Inversions',
    as.character(inv$contig),
    inv$position_start,
    inv$position_end,
    opacities = 0.7,
    values = inv$n_barcodes,
    minRadius = 0.78,
    maxRadius = 0.87,
    colors = color.palette["inversion"],
    labels = "Type: Inversion",
  )
}

# deletions
del <- sv[sv$type == "DEL",]
if(nrow(del) > 0){
  tracks <- tracks + BioCircosSNPTrack(
    'Deletions',
    as.character(del$contig),
    del$position_start,
    values = abs(del$position_end - del$position_start),
    minRadius = 0.63,
    maxRadius = 0.71,
    size = 1,
    shape = "rect",
    opacities = 0.7,
    colors = color.palette["deletion"],
    labels = "Type: Deletion"
  )
}


# duplications
dup <- sv[sv$type == "DUP",]
if(nrow(dup) > 0){
  tracks <- tracks + BioCircosArcTrack(
    'Duplications',
    as.character(dup$contig),
    dup$position_start,
    dup$position_end,
    values = dup$n_barcodes,
    opacities = 0.7,
    minRadius = 0.48,
    maxRadius = 0.57,
    colors = color.palette["duplication"],
    labels = "Type: Duplication",
  )
}

# breakends
bnd <- sv[sv$type == "BND",]
if(nrow(bnd) > 0){
  tracks <- tracks + BioCircosSNPTrack(
    'Breakends',
    as.character(bnd$contig),
    bnd$position_start,
    values = 0,
    minRadius = 0.33,
    maxRadius = 0.42,
    size = 1,
    shape = "rect",
    opacities = 0.7,
    colors = color.palette["breakend"],
    labels = "Type: Breakend"
  )
}

# loop to create backgrounds
for(i in c("inversion", "deletion", "duplication")){
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(i,"_background"),
    minRadius = radius, maxRadius = radius + 0.15,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius + 0.15
}

BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  SNPMouseOverCircleSize = 2,
  SNPMouseOverTooltipsHtml03 = "<br/>Length: ",
  width = "100%",
  height = "1000px",
  elementId = "circosplot" 
)